In [1]:
import sys
from pathlib import Path

SEITZ_REPO = Path.home() / "SeitzModel"
REFPROP_ROOT = SEITZ_REPO / "REFPROP"
REFPROP_LIBRARY = REFPROP_ROOT / "lib" / "librefprop.so.2.30"
ARGON_FILE = REFPROP_ROOT / "FLUIDS" / "ARGON.FLD"
MIXING_FILE = REFPROP_ROOT / "FLUIDS" / "HMX.BNC"

print("Python:", sys.executable)
print("Python version:", sys.version)
print("SeitzModel repository:", SEITZ_REPO)
print("REFPROP library:", REFPROP_LIBRARY)

assert REFPROP_LIBRARY.exists()
assert ARGON_FILE.exists()
assert MIXING_FILE.exists()

Python: /home/pnichols/.conda/envs/MD_GPU/bin/python
Python version: 3.13.13 | packaged by conda-forge | (main, Apr  8 2026, 02:00:33) [GCC 14.3.0]
SeitzModel repository: /home/pnichols/SeitzModel
REFPROP library: /home/pnichols/SeitzModel/REFPROP/lib/librefprop.so.2.30


In [2]:
from ctREFPROP.ctREFPROP import REFPROPFunctionLibrary

rp = REFPROPFunctionLibrary(str(REFPROP_LIBRARY))
rp.SETPATHdll(str(REFPROP_ROOT))

print("REFPROP version:", rp.RPVersion())

ierr, herr = rp.SETUPdll(
    1,
    str(ARGON_FILE),
    str(MIXING_FILE),
    "DEF",
)

print("SETUP ierr:", ierr)
print("SETUP message:", herr)

assert ierr == 0

REFPROP version: 10.0.0.02
SETUP ierr: 0
SETUP message: 


In [3]:
SATURATION_T_K = 100.0

saturation = rp.SATTdll(
    SATURATION_T_K,
    [1.0],
    1,
)

print(saturation)

assert saturation.ierr == 0
assert saturation.P > 0
assert saturation.Dl > saturation.Dv

SATTdlloutput(P=323.7671859443872, Dl=32.885204483444184, Dv=0.42201832719617405, x=array('d', [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]), y=array('d', [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]), ierr=0, herr='')


In [4]:
if str(REFPROP_ROOT) not in sys.path:
    sys.path.insert(0, str(REFPROP_ROOT))

from SeitzModel import SeitzModel

In [5]:
KPA_PER_BAR = 100.0
PSIA_PER_BAR = 14.5037738
CELSIUS_OFFSET = 273.15

def kelvin_to_celsius(temperature_K):
    return temperature_K - CELSIUS_OFFSET

def bar_to_psia(pressure_bar):
    return pressure_bar * PSIA_PER_BAR

In [9]:
temperature_K = 128.0
pressure_bar = 1.0

temperature_C = kelvin_to_celsius(temperature_K)
pressure_psia = bar_to_psia(pressure_bar)

seitz = SeitzModel(
    pressure_psia,
    temperature_C,
    "ARGON",
    [1.0],
)

print(seitz)

	                   P : 14.50377        psia
	                   T : -145.15000      C
	                   Q : 0.06577         keV
	                  Rc : 4.34549         nm
	                Pvap : 266.69947       psia
	                Pbub : 246.13672       psia
	              DeltaH : 114869.59946    J/kg
	              DeltaS : 897.41874       J/kg-K
	               Gibbs : -303254.52503   J/kg
	               Rho_l : 1.06338         g/cc
	               Rho_b : 0.08285         g/cc
	               Sigma : 0.00347         N/m
	            dSigmadT : -0.00019        N/m-K
	               P_err : -0.00000        psia
	           Rho_b_err : 0.00000         g/cc
	               G_err : 0.00045         J/kg
	           Sigma_err : 0.00000         N/m
	        dSigmadT_err : 0.00000         N/m-K
	               errID : None           
	   interp_iterations : 68.00000        count
	                  Lt : 0.14839         nm
	        DeltaE_spike : 421552.42300    J/kg
	        DeltaW_spik

In [10]:
result = {
    "temperature_K": temperature_K,
    "temperature_C": temperature_C,
    "pressure_bar": pressure_bar,
    "pressure_psia": pressure_psia,
    "Q_keV": seitz.Q,
    "Rc_nm": seitz.Rc,
    "Pvap_psia": seitz.Pvap,
    "Pbub_psia": seitz.Pbub,
    "Rho_l_g_cc": seitz.Rho_l,
    "Rho_b_g_cc": seitz.Rho_b,
    "Sigma_N_m": seitz.Sigma,
    "errID": seitz.errID,
    "P_err_psia": seitz.P_err,
    "G_err_J_kg": seitz.G_err,
    "iterations": seitz.interp_iterations,
}

result

{'temperature_K': 128.0,
 'temperature_C': -145.14999999999998,
 'pressure_bar': 1.0,
 'pressure_psia': 14.5037738,
 'Q_keV': 0.06576911833723598,
 'Rc_nm': 4.345492521228183,
 'Pvap_psia': 266.6994673170701,
 'Pbub_psia': 246.1367223254492,
 'Rho_l_g_cc': 1.063383937436861,
 'Rho_b_g_cc': 0.0828522254353328,
 'Sigma_N_m': 0.003469990849415876,
 'errID': None,
 'P_err_psia': -3.0298320220844755e-13,
 'G_err_J_kg': 0.00044664076333587627,
 'iterations': 68}

In [11]:
assert result["errID"] is None
assert result["Q_keV"] > 0
assert result["Rc_nm"] > 0
assert result["Pvap_psia"] > result["pressure_psia"]
assert result["Pbub_psia"] > result["pressure_psia"]
assert abs(result["P_err_psia"]) < 1e-6

print("REFPROP and SeitzModel validation passed.")

REFPROP and SeitzModel validation passed.
